# Qdrant vs CyborgDB Vec2Text Vulnerability Comparison

> **Last validated: 2026-03-11 | CyborgDB v0.15.0**

This notebook demonstrates the vec2text attack on both Qdrant and CyborgDB side-by-side.
 
### Attack Chain:
1. Sensitive texts → OpenAI embeddings → Store in BOTH Qdrant and CyborgDB
2. Extract embeddings from each database backend
3. Use vec2text to attempt reconstruction of original sensitive text
4. Compare results: Qdrant (vulnerable) vs CyborgDB (protected)

**NOTE**: You will need an OpenAI API key. Set it as your `OPENAI_API_KEY` environment variable or you will be prompted in cell `#1`.

In [ ]:
# 0. Install dependencies
# Uninstall sentence-transformers first to avoid conflict with older transformers
%pip uninstall -y sentence-transformers --quiet

# Install transformers 4.36.0 (before meta device check was added)
%pip install --quiet transformers==4.36.0

# Install other dependencies
%pip install --quiet vec2text==0.0.13 openai==2.26.0 numpy==2.0.2 qdrant-client==1.17.0 cyborgdb-core==0.15.0 psycopg2-binary==2.9.11 getpass4==0.0.14.1

# CPU-only torch
%pip install --quiet torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# 1. Set up OpenAI embedding & vec2text corrector models

import os
import vec2text
from openai import OpenAI
import numpy as np
import getpass

# Environment variable setup
def setup_env():
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1' 
    os.environ['MKL_NUM_THREADS'] = '1'
    os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS'] = '1'
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
setup_env()

# ANSI color codes for live demo
class Colors:
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    END = '\033[0m'

def print_colored(text, color="", bold=False):
    """Print colored text for demo"""
    prefix = Colors.BOLD if bold else ""
    prefix += getattr(Colors, color.upper(), "")
    print(f"{prefix}{text}{Colors.END}")

# OpenAI setup
embedding_model = "text-embedding-ada-002"
openai_api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter OPENAI_API_KEY: ")
os.environ["OPENAI_API_KEY"] = openai_api_key
openai_client = OpenAI()

# Load vec2text corrector for inversion
print(f"Loading vec2text corrector for OpenAI {embedding_model}...")
print("This may take a few minutes depending on your hardware...")
corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

In [ ]:
# 2. Install and Setup PostgreSQL

import subprocess
import platform
import time
import getpass

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# Detect OS
is_mac = platform.system() == "Darwin"
is_linux = platform.system() == "Linux"

# Install PostgreSQL if needed
if not run("which psql").stdout:
    if is_mac:
        run("brew install postgresql@14 >/dev/null 2>&1")
    elif is_linux:
        run("sudo apt-get update >/dev/null 2>&1")
        run("sudo apt-get install -y postgresql postgresql-contrib >/dev/null 2>&1")

# Always attempt to start the service
if is_mac:
    run("brew services start postgresql@14 >/dev/null 2>&1")
elif is_linux:
    run("sudo service postgresql start")

# Wait a bit for PostgreSQL to start
time.sleep(3)

# Setup connection parameters
POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_DB = "postgres"
POSTGRES_USER = getpass.getuser()
POSTGRES_PASSWORD = "password"

# Create user (if needed)
if is_linux:
    run(f"sudo -u postgres psql -c \"CREATE USER {POSTGRES_USER} WITH PASSWORD '{POSTGRES_PASSWORD}' SUPERUSER;\" 2>/dev/null")
else:
    run(f"psql postgres -c \"CREATE USER {POSTGRES_USER} WITH PASSWORD '{POSTGRES_PASSWORD}';\" 2>/dev/null")
    run(f"psql postgres -c \"ALTER USER {POSTGRES_USER} WITH SUPERUSER;\" 2>/dev/null")

print(f"✓ PostgreSQL ready: {POSTGRES_USER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

In [ ]:
# 3. Set up Qdrant and CyborgDB

import secrets
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from cyborgdb_core import Client, DBConfig, get_demo_api_key
import uuid

# Qdrant setup - Use local file-based storage (works everywhere, including Colab)
qdrant_storage_path = "./tmp/qdrant_storage"
# Delete existing collection if exists in path
if os.path.exists(qdrant_storage_path):
    import shutil
    shutil.rmtree(qdrant_storage_path)
    
qdrant_client = QdrantClient(path=qdrant_storage_path)
collection_name = "sensitive_docs"

try:
    qdrant_client.delete_collection(collection_name)
except:
    pass

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
print(f"✓ Qdrant ready: {qdrant_storage_path}")

# CyborgDB setup with PostgreSQL
postgres_connection_string = (
    f"host={POSTGRES_HOST} port={POSTGRES_PORT} dbname={POSTGRES_DB} "
    f"user={POSTGRES_USER} password={POSTGRES_PASSWORD}"
)

# Configure API key or demo API key (expires in 1 hour)
API_KEY = os.environ.get("CYBORGDB_API_KEY") or get_demo_api_key()
os.environ["CYBORGDB_API_KEY"] = API_KEY

# CyborgDB setup
cyborgdb_client = Client(
    API_KEY,
    DBConfig(location="postgres", connection_string=postgres_connection_string, table_name="index_comparison"),
    DBConfig(location="postgres", connection_string=postgres_connection_string, table_name="items_comparison"),
    DBConfig(location="postgres", connection_string=postgres_connection_string, table_name="config_comparison"),
)

# Create new CyborgDB index
cyborg_index = cyborgdb_client.create_index(
    index_name=f"comparison-demo-{uuid.uuid4()}",
    index_key=secrets.token_bytes(32),
)
print(f"✓ CyborgDB ready: {POSTGRES_USER}@{POSTGRES_HOST}/{POSTGRES_DB}")

In [ ]:
# 4. Define sensitive documents for demo

sensitive_documents = [
    # Financial/Identity Information
    "My social security number is 818-88-8118",
    "Password for admin account is SecurePass123!",
    
    # Medical Information
    "Patient diagnosed with Type 2 diabetes, prescribed metformin",
    "Blood test results show cholesterol level of 277 mg/dL",
    
    # Personal Information
    "John Doe lives at 1 World Trade Center, New York, NY 10007",
    "Phone number +1(212) 653-0688; email info@cyborg.co"
]

In [ ]:
# 5. Generate embeddings & store in BOTH databases

print("="*80)
print_colored("GENERATING EMBEDDINGS FROM SENSITIVE DOCUMENTS", bold=True)
print("="*80 + "\n")

def get_embeddings_openai(text_list):
    """Get embeddings from OpenAI API"""
    response = openai_client.embeddings.create(
        input=text_list,
        model=embedding_model,
        encoding_format="float",
    )
    return [e.embedding for e in response.data]

# Get embeddings (generate once, use for both databases)
embeddings = get_embeddings_openai(sensitive_documents)
print(f"Generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}\n")

# Store in Qdrant
points = [
    PointStruct(
        id=i,
        vector=embedding,
        payload={"text": doc, "type": "sensitive", "doc_num": i}
    )
    for i, (doc, embedding) in enumerate(zip(sensitive_documents, embeddings))
]
qdrant_client.upsert(collection_name=collection_name, points=points)
print("✓ Stored in Qdrant")

# Store in CyborgDB
items = [
    {"id": f"sensitive_doc_{i}", "vector": embedding, "contents": doc}
    for i, (doc, embedding) in enumerate(zip(sensitive_documents, embeddings))
]
cyborg_index.upsert(items)
print("✓ Stored in CyborgDB (encrypted)")

In [ ]:
# 6. Extract embeddings from Qdrant SQLite backend

import sqlite3
import pickle
import io

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM QDRANT", bold=True)
print("="*80 + "\n")

# Custom unpickler that doesn't require qdrant_client module
class CustomUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Return a dummy class for anything we can't import
        return type(name, (), {})

# Connect to Qdrant's SQLite database
db_path = os.path.join(qdrant_storage_path, "collection", collection_name, "storage.sqlite")
print(f"Connecting to: {db_path}\n")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Query points table
cursor.execute("SELECT id, point FROM points")
rows = cursor.fetchall()

qdrant_extracted = []

for row in rows:
    point_id = row[0]
    point_blob = row[1]
    
    if point_blob:
        try:
            # Unpickle the Qdrant point data
            data = CustomUnpickler(io.BytesIO(point_blob)).load()
            
            # Extract vector - check nested __dict__ structure
            vector = None
            if hasattr(data, '__dict__'):
                # Try data.__dict__['__dict__']['vector']
                if '__dict__' in data.__dict__ and 'vector' in data.__dict__['__dict__']:
                    vector = data.__dict__['__dict__']['vector']
                # Try data.__dict__['vector']
                elif 'vector' in data.__dict__:
                    vector = data.__dict__['vector']
            
            if vector:
                qdrant_extracted.append(vector)
                norm = np.linalg.norm(vector)
                print_colored(f"Extracted Qdrant vector {len(qdrant_extracted)}: id={point_id}, dim={len(vector)}, norm={norm:.4f}", "RED")
        except Exception as e:
            print(f"Error extracting: {e}")
            continue

conn.close()

print(f"\n✓ Extracted {len(qdrant_extracted)} embeddings from Qdrant SQLite (plaintext in pickle!)")

In [ ]:
# 7. Extract embeddings from CyborgDB PostgreSQL backend

import numpy as np
import psycopg2
from psycopg2.extras import RealDictCursor

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM CYBORGDB", bold=True)
print("="*80 + "\n")

# Connect to CyborgDB's PostgreSQL database
pg_client = psycopg2.connect(
    host=POSTGRES_HOST, database=POSTGRES_DB,
    user=POSTGRES_USER, password=POSTGRES_PASSWORD, port=POSTGRES_PORT
)
cursor = pg_client.cursor(cursor_factory=RealDictCursor)

# Get data from CyborgDB tables
cursor.execute("SELECT key, value FROM index_comparison;")
rows = cursor.fetchall()

cyborg_extracted = []
for i, row in enumerate(rows):
    value = row['value']
    if value is not None:
        bytes_data = value.tobytes() if hasattr(value, 'tobytes') else bytes(value)
        
        if len(bytes_data) < 1536 * 4:
            continue
        
        # Calculate entropy to check if encrypted
        entropy = -sum(
            (bytes_data.count(byte) / len(bytes_data)) * np.log2(bytes_data.count(byte) / len(bytes_data)) 
            for byte in set(bytes_data)
        )
        is_encrypted = entropy > 7.8 and len(bytes_data) != 1536 * 4
        
        if is_encrypted:
            bytes_data = bytes_data[:1536 * 4]
        
        embedding_values = np.frombuffer(bytes_data, dtype=np.float32)
        cyborg_extracted.append(embedding_values.tolist())
        
        status = "ENCRYPTED" if is_encrypted else "PLAINTEXT"
        color = "GREEN" if is_encrypted else "RED"
        print_colored(f"Extracted CyborgDB embedding {len(cyborg_extracted)}: {row['key']}, {len(embedding_values)} dimensions [{status}]", color)

print(f"\n✓ Extracted {len(cyborg_extracted)} embeddings from CyborgDB")

In [ ]:
# 8. Define inversion function

import time
import torch

def run_inversion_attack(extracted_embeddings, db_name, color=""):
    """Run vec2text inversion attack on embeddings from a database"""
    
    print("\n" + "="*80)
    print_colored(f"{db_name.upper()}: RUNNING EMBEDDING INVERSION ATTACK", color, bold=True)
    print("="*80)
    
    # Convert to tensor
    embeddings_tensor = torch.tensor(extracted_embeddings, dtype=torch.float32)
    if torch.backends.mps.is_available():
        embeddings_tensor = embeddings_tensor.to('mps')
    elif torch.cuda.is_available():
        embeddings_tensor = embeddings_tensor.cuda()
    
    results = []
    
    for i, (original_doc, embedding) in enumerate(zip(sensitive_documents, embeddings_tensor)):
        print_colored(f"\n[{db_name}] Document #{i+1}:", bold=True)
        print(f"Original:      \"{original_doc}\"")
        
        start_time = time.time()
        reconstructed_list = vec2text.invert_embeddings(
            embeddings=embedding.unsqueeze(0),
            corrector=corrector,
            num_steps=4,
            sequence_beam_width=1,
        )
        reconstructed = reconstructed_list[0]
        inversion_time = time.time() - start_time
        
        print(f"Reconstructed: \"{reconstructed}\"")
        
        # Calculate similarity
        if len(reconstructed) > 0:
            orig_emb_cpu = embedding.cpu()
            new_emb = get_embeddings_openai([reconstructed])[0]
            new_emb_tensor = torch.tensor(new_emb)
            similarity = torch.nn.functional.cosine_similarity(orig_emb_cpu, new_emb_tensor, dim=0).item()
        else:
            similarity = 0
        
        exact_match = original_doc.lower().strip() == reconstructed.lower().strip()
        if exact_match:
            print_colored(f"Exact match!", "RED", bold=True)
        
        sim_color = "RED" if similarity > 0.99 else "YELLOW" if similarity > 0.95 else "GREEN"
        print_colored(f"Similarity: {similarity:.4f}", sim_color, bold=True)
        print(f"Time: {inversion_time:.2f}s")
        
        results.append({
            'original': original_doc,
            'reconstructed': reconstructed,
            'similarity': similarity,
            'time': inversion_time,
            'exact_match': exact_match,
        })
    
    return results

In [ ]:
# 9a. Run attacks on both databases

# Attack Qdrant
qdrant_results = run_inversion_attack(qdrant_extracted, "Qdrant", "RED")

# Attack CyborgDB  
cyborg_results = run_inversion_attack(cyborg_extracted, "CyborgDB", "GREEN")

In [ ]:
# 9b. Run attack on CyborgDB

# Attack CyborgDB  
cyborg_results = run_inversion_attack(cyborg_extracted, "CyborgDB", "GREEN")

In [ ]:
# 10. Side-by-Side Comparison Summary

print("\n" + "="*80)
print_colored("COMPARISON SUMMARY: Qdrant vs CyborgDB", bold=True)
print("="*80 + "\n")

def calc_stats(results):
    total = len(results)
    exact = sum(1 for r in results if r['exact_match'])
    high_sim = sum(1 for r in results if r['similarity'] > 0.95)
    avg_sim = np.mean([r['similarity'] for r in results])
    avg_time = np.mean([r['time'] for r in results])
    return {
        'total': total,
        'exact': exact,
        'high_sim': high_sim,
        'avg_sim': avg_sim,
        'avg_time': avg_time
    }

qdrant_stats = calc_stats(qdrant_results)
cyborg_stats = calc_stats(cyborg_results)

print(f"{'Metric':<35} {'Qdrant':>15} {'CyborgDB':>15}")
print("="*80)
print(f"{'Total documents':<35} {qdrant_stats['total']:>15} {cyborg_stats['total']:>15}")
print(f"{'Exact reconstructions':<35} {qdrant_stats['exact']:>15} {cyborg_stats['exact']:>15}")
print(f"{'High similarity (>95%)':<35} {qdrant_stats['high_sim']:>15} {cyborg_stats['high_sim']:>15}")
print(f"{'Average similarity':<35} {qdrant_stats['avg_sim']*100:>14.2f}% {cyborg_stats['avg_sim']*100:>14.2f}%")
print(f"{'Average inversion time':<35} {qdrant_stats['avg_time']:>14.2f}s {cyborg_stats['avg_time']:>14.2f}s")

print("\n" + "="*80)
print_colored("CONCLUSION", bold=True)
print("="*80)

if qdrant_stats['avg_sim'] > 0.95:
    print_colored("⚠ Qdrant: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ Qdrant: Protected - Low similarity indicates failed reconstruction", "GREEN", bold=True)

if cyborg_stats['avg_sim'] > 0.95:
    print_colored("⚠ CyborgDB: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ CyborgDB: PROTECTED - Low similarity indicates failed reconstruction", "GREEN", bold=True)

In [ ]:
# 11. Cleanup

# Delete Qdrant collection
try:
    qdrant_client.delete_collection(collection_name)
    print("✓ Qdrant collection deleted")
except:
    pass

# Close connections
pg_client.close()
cyborg_index.delete_index()
print("✓ CyborgDB cleaned up")